# NIST Parylene-C Palace CPW benchmark

**Objective.** Model the complete 6.5 mm, air-filled H2O-chip CPW in Palace and compare the full complex S matrix first to the published ANSYS Q2D cascade, then to the calibrated measurement.

**Hypothesis.** The 0.2 µm Pt model should approach Q2D after matching all nine line sections. The capped screening run uses explicit 50-ohm CPW lumped ports; their termination reflection must be separated from device/model error. The separately labelled 0.405 µm Pt case is the appropriate measurement comparison.

The default execution validates the stored references and constructs the model without meshing or submitting. Set `GSIM_EIC_RUN_MESH=1` for local meshing or `GSIM_EIC_RUN_CLOUD=1` for the opt-in three-frequency cloud screening run. Every Palace execution is blocked unless the finalized mesh contains at most 110,000 tetrahedra.

In [ ]:
# Copyright 2026 GDSFactory

from __future__ import annotations

import importlib.metadata
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from gsim.palace.benchmarks import (
    from_palace_sparams,
    interpolate_sparameters,
    maximum_singular_value,
    power_loss_fraction,
    reciprocity_error,
    sparameter_error_summary,
)
from gsim.palace.benchmarks.eic_nist import (
    BOTTOM_CONDUCTOR_THICKNESS_UM,
    NIST_BULK_MESH_SIZE_UM,
    NIST_CAPPED_MESH_SIZE_UM,
    NIST_MESH_ALGORITHM_3D,
    NIST_MAX_TETRAHEDRA,
    NIST_MEASUREMENT_DOI,
    NIST_SIMULATION_DOI,
    build_nist_component,
    build_nist_stack,
    cascade_nist_rlcg,
    load_nist_air_reference,
    make_nist_simulation,
    nist_section_edges_um,
    require_nist_tetrahedron_budget,
)


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from within the gsim repository")


repo_root = find_repo_root()
data_dir = repo_root / "tests" / "data" / "eic" / "nist"
run_cloud = os.getenv("GSIM_EIC_RUN_CLOUD") == "1"
run_mesh = os.getenv("GSIM_EIC_RUN_MESH") == "1" or run_cloud
model_case = os.getenv("GSIM_EIC_NIST_CASE", "ansys").lower()
numerical_order = int(os.getenv("GSIM_EIC_NIST_ORDER", "2"))
if model_case not in {"ansys", "measurement"}:
    raise ValueError("GSIM_EIC_NIST_CASE must be 'ansys' or 'measurement'")
if numerical_order not in {1, 2}:
    raise ValueError("GSIM_EIC_NIST_ORDER must be 1 or 2")
print(
    {
        "gsim": importlib.metadata.version("gsim"),
        "case": model_case,
        "mesh": run_mesh,
        "cloud": run_cloud,
        "order": numerical_order,
    }
)

## Published references

The inputs are from NIST datasets [10.18434/mds2-2817](https://doi.org/10.18434/mds2-2817) (simulation) and [10.18434/mds2-2808](https://doi.org/10.18434/mds2-2808) (measurement). The final RLCG CSVs reproduce `constructSfromSim.m`. The stored MAT result was created earlier than those CSV revisions, so the regression records their small, bounded difference instead of claiming bit identity.

In [ ]:
reference = load_nist_air_reference(data_dir / "air_data_vs_sim.npz")
reconstructed = cascade_nist_rlcg(data_dir)
archive_revision_delta = np.abs(reconstructed.s - reference.simulation.s)
np.testing.assert_allclose(np.max(archive_revision_delta), 0.0017645356, rtol=1e-6)

print("simulation DOI:", NIST_SIMULATION_DOI)
print("measurement DOI:", NIST_MEASUREMENT_DOI)
print(f"final-CSV vs stored-MAT max |delta S|={np.max(archive_revision_delta):.8f}")
print(f"Q2D reciprocity error={reciprocity_error(reference.simulation):.3g}")
print(f"Q2D largest singular value={maximum_singular_value(reference.simulation):.9f}")
print(f"measurement reciprocity error={reciprocity_error(reference.measurement):.4g}")
print(
    f"measurement largest singular value={maximum_singular_value(reference.measurement):.6f}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
for data, style, label_prefix in (
    (reference.simulation, "-", "Q2D"),
    (reference.measurement, "--", "measured"),
):
    band = (data.frequency_hz >= 0.1e9) & (data.frequency_hz <= 20e9)
    frequency_ghz = data.frequency_hz[band] / 1e9
    for axis, (label, row, column) in zip(
        axes, (("S11", 0, 0), ("S21", 1, 0)), strict=True
    ):
        axis.plot(
            frequency_ghz,
            20 * np.log10(np.abs(data.s[band, row, column])),
            style,
            label=f"{label_prefix} {label}",
        )
for axis, title in zip(axes, ("Reflection", "Transmission"), strict=True):
    axis.set(xlabel="Frequency (GHz)", ylabel="Magnitude (dB)", title=title)
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()

## Complete geometry and material cases

The nine-section sequence is `NP–YP–AirPDMS–PDMS–AirChannel–PDMS–AirPDMS–YP–NP`. The centered 213 µm air channel appears in both AirPDMS transitions and the central channel, overriding the surrounding PDMS. The CPW is 50/2.5/200 µm signal/gap/ground on 500 µm fused silica, with 6.67 µm Parylene C.

`ansys` selects the source model's 0.2 µm Pt and Q2D as its target. `measurement` selects the measured 0.405 µm Pt thickness and calibrated data; set it with `GSIM_EIC_NIST_CASE=measurement`.

In [ ]:
component = build_nist_component()
edges = nist_section_edges_um()
ansys_stack = build_nist_stack(platinum_thickness_um=0.2)
measurement_stack = build_nist_stack(platinum_thickness_um=0.405)
assert ansys_stack.validate_stack().valid
assert measurement_stack.validate_stack().valid
np.testing.assert_allclose(component.bbox_np(), [[-3250.25, -682.5], [3250.25, 682.5]])

print("section edges (um):", edges)
print("total line length (um):", edges[-1] - edges[0])
print("ANSYS Pt thickness (um):", ansys_stack.layers["platinum"].thickness)
print("measured Pt thickness (um):", measurement_stack.layers["platinum"].thickness)

In [ ]:
platinum_thickness_um = 0.2 if model_case == "ansys" else 0.405
target_reference = (
    reference.simulation if model_case == "ansys" else reference.measurement
)
simulation = make_nist_simulation(
    repo_root / f"palace-sim-eic-nist-{model_case}",
    platinum_thickness_um=platinum_thickness_um,
    num_points=3,
    adaptive_tol=0.0,
    adaptive_max_samples=1,
    numerical_order=numerical_order,
)
assert simulation.validate_config().valid
palace_result = None
mesh_result = None
print(
    f"Configured {model_case} case: Pt={platinum_thickness_um} um, two 50-ohm CPW lumped ports."
)

## Optional mesh and run

The opt-in mesh retains the full 6.5005 mm by 1.365 mm physical device, adds 50 µm longitudinal and 100 µm lateral airbox margins, and keeps absorption on outer air only. Same-coefficient geometry partitions force four 0.625 µm transverse spans across every 2.5 µm gap at the metal plane while allowing wavelength-scale longitudinal and bulk elements. HXT tetrahedralization uses 150 µm refined and 200 µm bulk targets. `planar_conductors=False` keeps Pt finite; the order-2, three-frequency diagnostic uses two excited 50-ohm CPW lumped ports and a `1e-8` linear tolerance. Do not interpret material loss until the same-mesh PEC control, reciprocity, passivity, port, and mesh convergence have been checked.

In [ ]:
if run_mesh:
    mesh_result = simulation.mesh(
        preset="coarse",
        refined_mesh_size=NIST_CAPPED_MESH_SIZE_UM,
        max_mesh_size=NIST_BULK_MESH_SIZE_UM,
        algorithm_3d=NIST_MESH_ALGORITHM_3D,
        planar_conductors=False,
        verbose=False,
    )
    tetrahedra = require_nist_tetrahedron_budget(mesh_result)
    config_path = simulation.write_config()
    validation = simulation.validate_mesh()
    config = json.loads(config_path.read_text())
    conductivity_boundaries = config["Boundaries"]["Conductivity"]
    lumped_ports = config["Boundaries"]["LumpedPort"]
    assert len(conductivity_boundaries) == 4
    np.testing.assert_allclose(
        [entry["Thickness"] for entry in conductivity_boundaries],
        [
            platinum_thickness_um,
            platinum_thickness_um,
            BOTTOM_CONDUCTOR_THICKNESS_UM,
            BOTTOM_CONDUCTOR_THICKNESS_UM,
        ],
    )
    assert len(lumped_ports) == 2
    assert all(port["R"] == 50.0 for port in lumped_ports)
    assert all(len(port["Elements"]) == 2 for port in lumped_ports)
    assert config["Boundaries"]["Absorbing"]["Order"] == 1
    assert config["Solver"]["Order"] == numerical_order
    assert config["Solver"]["Linear"]["Tol"] == 1e-8
    assert tetrahedra <= NIST_MAX_TETRAHEDRA
    print(validation)
    print(f"tetrahedra: {tetrahedra:,} / {NIST_MAX_TETRAHEDRA:,}")
    print(mesh_result.mesh_stats)
    print("conductivity boundaries:", conductivity_boundaries)
else:
    print("Mesh skipped; set GSIM_EIC_RUN_MESH=1 to reproduce local validation.")

In [ ]:
if run_cloud:
    require_nist_tetrahedron_budget(mesh_result)
    palace_result = simulation.run(check_cache=True, verbose="status")
    print("cloud job id:", simulation._job_id)
else:
    print("Cloud submission skipped; set GSIM_EIC_RUN_CLOUD=1 explicitly.")

In [ ]:
if palace_result is not None:
    palace = from_palace_sparams(palace_result)
    aligned_reference = interpolate_sparameters(target_reference, palace.frequency_hz)
    parity = sparameter_error_summary(aligned_reference, palace)
    print(json.dumps(parity, indent=2))
    print(f"Palace reciprocity error={reciprocity_error(palace):.3g}")
    print(f"Palace largest singular value={maximum_singular_value(palace):.6f}")
    print(
        "Palace unscattered power range:",
        power_loss_fraction(palace).min(),
        power_loss_fraction(palace).max(),
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
    for data, style, prefix in (
        (aligned_reference, "-", "reference"),
        (palace, "--", "Palace"),
    ):
        for label, row, column in (
            ("S11", 0, 0),
            ("S12", 0, 1),
            ("S21", 1, 0),
            ("S22", 1, 1),
        ):
            values = data.s[:, row, column]
            axes[0].plot(
                data.frequency_hz / 1e9,
                20 * np.log10(np.abs(values)),
                style,
                label=f"{prefix} {label}",
            )
            axes[1].plot(
                data.frequency_hz / 1e9,
                np.unwrap(np.angle(values)) * 180 / np.pi,
                style,
                label=f"{prefix} {label}",
            )
    axes[0].set(xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
    axes[1].set(xlabel="Frequency (GHz)", ylabel="Unwrapped phase (degrees)")
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend(ncol=2)
    fig.tight_layout()
else:
    print("Parity table will be emitted after an opt-in Palace run completes.")

## Findings and decision log

- The curated arrays preserve all four complex entries and both published frequency grids.
- The final archive CSVs reproduce the older stored Q2D cascade to a maximum `|delta S|` of 0.001765; provenance dates explain the nonzero revision delta.
- The Palace model is the complete 6.5005 mm measured section sequence, not a shortened or periodic surrogate.
- The NIST AEDT source includes a 50 um platinum back conductor below the 500 um quartz substrate; the 3D model now includes it. The corrected finite-Pt jobs at 72,768 and 80,146 tetrahedra remained tens of decibels below the Q2D S21 target at high frequency. Raising the same 80,146-tet mesh from order 1 to order 2 improved S21 by 10–21 dB, proving that the order-1 result was not converged.
- Official Palace v0.17 CPW guidance supports the two-element lumped-port topology and uses order 2. The diagnostic therefore uses order 2, a `1e-8` linear tolerance, a uniform 0.1/10.05/20 GHz sweep, and first-order absorption on outer air only.
- The complete 3D device has 50 um longitudinal and 100 um lateral air margins. Only `air__None` is absorbing; patterned Parylene-C, PDMS, and quartz no longer touch the artificial boundary.
- Same-coefficient mesh partitions force four 0.625 um transverse spans across every 2.5 um CPW gap. HXT with 150 um refined and 200 um bulk targets produces 9,277 nodes and 50,217 tetrahedra, zero invalid SICN elements, and min/mean SICN 0.001617/0.5582.
- Two localized numeric-wave-port jobs with 78,266 tetrahedra did not leave the boundary eigensolve and were cancelled. Their faces omitted most of the substrate and back conductor, so those inputs are rejected rather than interpreted as a Palace limitation.
- The same-mesh finite-Pt and top-Pt PEC controls completed in 10m08s and 10m13s. At 0.1/10.05/20 GHz, NIST Q2D S21 is -5.15/-9.66/-12.96 dB, finite Pt is -5.24/-29.59/-43.82 dB, and PEC is -0.03/-1.08/-2.48 dB. The 5.21/28.51/41.34 dB finite-to-PEC difference localizes the excess loss to the metal treatment, while PEC remains an under-lossy diagnostic only.
- A zero-geometric-thickness 200 nm Pt sheet completed in 9m34s on 47,844 tetrahedra and gave S21 -5.26/-29.23/-39.38 dB. Replacing `Conductivity` with an independent 0.5531 ohm/square `Impedance` boundary changed S21 by at most 0.067 dB, ruling out a finite-thickness-formula-specific defect.
- The 87,475-tetrahedron HXT 150/150 sheet run completed in 17m48s and gave S21 -5.25/-28.65/-46.16 dB. Relative to the 47,844-tet sheet result, the changes are +0.012/+0.581/-6.779 dB. The capped full-chip model is not mesh-converged at 20 GHz and remains about 19 dB below NIST at 10.05 GHz; do not launch a production sweep or claim parity.
- All completed finite-Pt diagnostics are reciprocal and passive to the reported solver accuracy. Several linear/error-indicator solves stop above the requested tolerance, so solver convergence remains an additional caveat rather than an explanation for the tens-of-decibels gap.
- Released `gsim` 0.4.0 cannot generate this combined geometry without the worktree's meshing fixes, but the cloud executes uploaded `config.json` and `palace.msh` directly. The hard 110,000-tetrahedron gate is checked after meshing, immediately before submission, and by recounting the final staged mesh. The next investigation should isolate one uniform NIST cross-section in a genuine 3D Palace line before returning to the complete nine-section chip.